## Parameters

Modify this section to provide the correct path to the data.

In [1]:
import anndata as ad
import dotenv
import os

dotenv.load_dotenv()

pert_col = "target_gene"
control = "non-targeting"
adata = ad.read_h5ad(os.getenv("DATA_PATH"), backed=True) # FIXME

## Calculate the mean expression

In [2]:
from scipy.sparse import csr_matrix
import numpy as np


def get_grouped_mean_var(adata: ad.AnnData, group_column: str="target_gene") -> tuple[np.ndarray, np.ndarray]:
    """Get mean and variance of each group"""
    X = adata.X
    labels = np.asarray(adata.obs[group_column])
    unique_labels, inverse = np.unique(labels, return_inverse=True)
    n_groups = len(unique_labels)
    group_sizes = np.bincount(inverse)

    # Mean
    one_hot = csr_matrix(
        (np.ones_like(inverse), (inverse, np.arange(len(inverse)))),
        shape=(n_groups, X.shape[0])
    )
    group_sums = one_hot @ X  # (n_groups, n_genes)
    group_means = group_sums.toarray() / group_sizes[:, None]

    return group_means, unique_labels


perts = adata.obs[pert_col].unique()
perts = perts[perts != control]
train_adata = adata[adata.obs[pert_col].isin(perts)].to_memory()
means, eval_perts = get_grouped_mean_var(train_adata) # Get lambda
means

array([[0.00000000e+00, 1.30594901e+00, 2.53824363e-01, ...,
        7.19546742e-02, 2.21586402e+00, 1.43852691e+00],
       [7.14285714e-04, 1.39928571e+00, 2.37142857e-01, ...,
        8.85714286e-02, 2.62714286e+00, 1.61642857e+00],
       [1.06609808e-03, 1.46055437e+00, 2.83582090e-01, ...,
        1.08742004e-01, 2.91364606e+00, 1.87420043e+00],
       ...,
       [2.39234450e-03, 1.45454545e+00, 2.70334928e-01, ...,
        9.09090909e-02, 3.35885167e+00, 2.35645933e+00],
       [0.00000000e+00, 1.39397742e+00, 2.29611041e-01, ...,
        9.78670013e-02, 2.54956085e+00, 1.58971142e+00],
       [2.34741784e-03, 1.40845070e+00, 2.55868545e-01, ...,
        9.38967136e-02, 2.78403756e+00, 1.78286385e+00]],
      shape=(150, 18080))

## Simulate with Poisson distribution

In [3]:
def simulate_with_poisson(means: np.ndarray, sizes: np.ndarray, seed: int=2409) -> tuple[np.ndarray, np.ndarray]:
    """Simulate with Poisson distribution"""
    rng = np.random.default_rng(seed)
    n_groups, n_genes = means.shape

    # Pre-allocate
    total_cells = int(np.sum(sizes))
    X_sim = np.empty((total_cells, n_genes), dtype=np.float32)
    labels_sim = np.empty(total_cells, dtype=int)

    start = 0
    for g in range(n_groups):
        n = sizes[g]
        stop = start + n
        mu = means[g]
        X_sim[start:stop] = rng.poisson(mu, size=(n, n_genes))
        labels_sim[start:stop] = g
        start = stop
    return X_sim, labels_sim

size_map = adata.obs[pert_col].value_counts().to_dict()
sizes = [size_map[p] for p in eval_perts]
X_sim, labels_sim = simulate_with_poisson(means, sizes) # Poisson
X_sim.shape

(183097, 18080)

## Evaluation with DES, MAE, and PDS

The notebook was using cell-eval 0.6.5. Please refer to `pyproject.toml` for more information.

In [4]:
from cell_eval import MetricsEvaluator
import pandas as pd

# Turn off functools exception
import warnings
warnings.filterwarnings("ignore", module="functools")

sim = ad.AnnData(
    X_sim,
    obs=pd.DataFrame({pert_col: eval_perts[labels_sim]}),
    var=adata.var,
)
ctrl = adata[adata.obs[pert_col] == control].to_memory()
eval = MetricsEvaluator(
    adata_real=adata.to_memory(),
    adata_pred=ad.concat([sim, ctrl]),
    pert_col=pert_col,
    control_pert=control,
)
results, agg_results = eval.compute(profile="vcc")
agg_results

INFO:cell_eval.utils:Data appears to be integer counts (no decimal values detected)
INFO:cell_eval._evaluator:Discovered integer data for real. Converting to norm-log.
INFO:cell_eval.utils:Data appears to be integer counts (no decimal values detected)
INFO:cell_eval._evaluator:Discovered integer data for pred. Converting to norm-log.
INFO:cell_eval._evaluator:Computing DE for real data
INFO:cell_eval._evaluator:Using the following pdex kwargs: {'reference': 'non-targeting', 'groupby_key': 'target_gene', 'num_workers': 96, 'batch_size': 100, 'metric': 'wilcoxon', 'is_log1p': True, 'as_polars': True}
INFO:pdex._single_cell:Log1p status: True
INFO:pdex._single_cell:Precomputing masks for each target gene
Identifying target masks: 100%|██████████| 151/151 [00:00<00:00, 2017.84it/s]
INFO:pdex._single_cell:Precomputing variable indices for each feature
Identifying variable indices: 100%|██████████| 18080/18080 [00:00<00:00, 3064579.36it/s]
INFO:pdex._single_cell:Creating shared memory memory

statistic,overlap_at_N,mae,discrimination_score_l1
str,f64,f64,f64
"""count""",150.0,150.0,150.0
"""null_count""",0.0,0.0,0.0
"""mean""",0.417927,0.042142,0.999644
"""std""",0.214427,0.003199,0.002012
"""min""",0.14534,0.039586,0.98
"""25%""",0.24,0.040727,1.0
"""50%""",0.365103,0.041152,1.0
"""75%""",0.588176,0.042185,1.0
"""max""",0.876474,0.067923,1.0
